# 02 — Chunking Strategies

**Series:** RAG for Financial & Legal Documents  
**Article:** «Как разбить документы на куски?»

---

## The naive question

> *Just split every 500 characters, right?*

This seems reasonable. But consider:

```
...Revenue for 2023 was $42.5 million, compared to $36.0 million in 202
                                                           ↑ SPLIT HERE
2. The enterprise segment accounted for 65% of total revenue...
```

The first chunk ends with `202` — half of a year. The second chunk begins with `2.` — a sentence fragment. Neither chunk is useful on its own.

**Chunking is the most underrated decision in RAG.** Poor chunking breaks retrieval even with a perfect embedding model and a great LLM.

---

## What you will learn

1. **Fixed-size chunking** — simple, fast, and its failure modes
2. **Semantic chunking** — split by meaning, not by character count
3. **Document-aware chunking** — respect headings and section structure
4. **LangChain equivalents** — same results in fewer lines
5. **Comparison** — which strategy wins on which document type

---

## Why chunk size matters: a mental model

Think of retrieval as a spotlight:
- **Too small a chunk (< 100 chars):** the spotlight is tiny. You find the exact sentence, but it has no context. The LLM can't answer from it.
- **Too large a chunk (> 3000 chars):** the spotlight covers the whole room. You retrieve the right section, but it's full of noise from unrelated paragraphs.
- **Sweet spot (300–1500 chars):** enough context to answer, small enough to be specific.

The right size depends on your documents AND your queries. Financial Q&A needs precise numbers → smaller chunks. Policy interpretation needs surrounding context → larger chunks.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import settings
print(f'Project root : {project_root}')

# We'll use Acme Corp's financial report as our test document
from src.from_scratch.ingestion.parsers.pdf_parser import parse_pdf

pdf_path = project_root / 'data/raw/clients/acme_corp/financial_report_2023.pdf'
parsed = parse_pdf(pdf_path, metadata={'client_id': 'acme_corp', 'doc_type': 'financial_report', 'year': 2023})
text = parsed.full_text
print(f'Document: {pdf_path.name}')
print(f'Pages: {parsed.num_pages}  |  Total chars: {len(text)}  |  Tables: {len(parsed.all_tables)}')

## Part 1 — Fixed-Size Chunking (From Scratch)

Split text at fixed character positions with a configurable overlap.

**Key parameters:**
- `chunk_size`: maximum characters per chunk (maps roughly to `chunk_size / 4` tokens)
- `chunk_overlap`: characters repeated at the start of the next chunk

**How overlap works:**
```
chunk_size=20, overlap=5
Text:   |AAAAAAAAAAAAAAABBBBB|BBBBBBBBBBBBBBBCCCCC|
Chunk1: |AAAAAAAAAAAAAAAAAAAA|
Chunk2:              |BBBBBBBBBBBBBBBBBBBBB|
Chunk3:                           |CCCCCCCCCCCCCCCCCCCC|
                      ^--- 5 chars of B appear in both chunk1 and chunk2
```

In [ ]:
from src.from_scratch.ingestion.chunking.fixed_size import split_fixed_size

chunks_fixed = split_fixed_size(
    text=text,
    chunk_size=512,
    chunk_overlap=64,
    metadata={'client_id': 'acme_corp', 'doc_type': 'financial_report'},
)

print(f'Total chunks : {len(chunks_fixed)}')
print(f'Avg size     : {sum(c.char_length for c in chunks_fixed)/len(chunks_fixed):.0f} chars')
print(f'Min size     : {min(c.char_length for c in chunks_fixed)} chars')
print(f'Max size     : {max(c.char_length for c in chunks_fixed)} chars')
print()
print('--- Chunk 0 ---')
print(repr(chunks_fixed[0]))
print(chunks_fixed[0].text)
print()
print('--- Chunk 1 (overlap with chunk 0) ---')
print(chunks_fixed[1].text[:120])

In [ ]:
# Demonstrate the failure mode: mid-sentence cuts
print('=== Fixed-size failure mode: mid-sentence cuts ===')
print()
for i, chunk in enumerate(chunks_fixed[:5]):
    first_line = chunk.text.split('\n')[0][:80]
    last_chars = chunk.text[-40:].replace('\n', ' ')
    print(f'Chunk {i}: ends with: ...{last_chars!r}')
    print(f'         starts with: {chunk.text[:60].replace(chr(10), " ")!r}')
    print()

## Part 2 — Semantic Chunking (From Scratch)

Instead of splitting at fixed positions, we split where the *meaning* changes.

**Algorithm:**
1. Split text into sentences
2. Embed each sentence with `sentence-transformers/all-MiniLM-L6-v2`
3. Compute cosine similarity between adjacent sentence embeddings
4. Insert a chunk boundary where similarity drops below the threshold
5. Merge sentences within each boundary into one chunk

```
Sentences:  [S1] [S2] [S3]  [S4] [S5]  [S6]
Similarity:    0.82  0.79  0.31  0.85  0.77
                              ↑
                        threshold=0.5
                        SPLIT HERE

Chunk 1: S1 + S2 + S3  (all about revenue)
Chunk 2: S4 + S5 + S6  (all about expenses)
```

**Note:** First run downloads the model (~90 MB). Subsequent runs use cache.

In [ ]:
from src.from_scratch.ingestion.chunking.semantic import split_semantic

chunks_semantic = split_semantic(
    text=text,
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    similarity_threshold=0.5,
    min_chunk_sentences=2,
    max_chunk_sentences=15,
    metadata={'client_id': 'acme_corp', 'doc_type': 'financial_report'},
)

print(f'Total chunks : {len(chunks_semantic)}')
print(f'Avg size     : {sum(c.char_length for c in chunks_semantic)/len(chunks_semantic):.0f} chars')
print(f'Avg sentences: {sum(len(c.sentences) for c in chunks_semantic)/len(chunks_semantic):.1f}')
print()
for i, c in enumerate(chunks_semantic[:3]):
    print(f'--- Semantic Chunk {i} ({len(c.sentences)} sentences, avg_sim={c.avg_similarity:.2f}) ---')
    print(c.text[:300])
    print()

## Part 3 — Document-Aware Chunking (From Scratch)

Use the document's own structure as chunk boundaries.

**Why this matters for financial documents:**
- A financial report has sections: "Revenue", "Operating Expenses", "Risk Factors"
- Each section should be a chunk — mixing sections creates noise
- The heading is crucial: `"it grew 18%"` means nothing without `"## Revenue"`

Our `docx_parser` preserves headings as markdown (`# H1`, `## H2`).  
Our `document_aware` chunker uses those markers as split points and prepends them to each chunk via `chunk.text_with_heading`.

```
Document:           Chunk with heading:
## Revenue          ## Revenue
It grew 18%    →    It grew 18%
...                 ...
```

In [ ]:
from src.from_scratch.ingestion.chunking.document_aware import split_document_aware

chunks_docaware = split_document_aware(
    text=text,
    max_chunk_size=1500,
    min_chunk_size=100,
    metadata={'client_id': 'acme_corp', 'doc_type': 'financial_report'},
)

print(f'Total chunks : {len(chunks_docaware)}')
print(f'Avg size     : {sum(c.char_length for c in chunks_docaware)/len(chunks_docaware):.0f} chars')
print()
print('Chunk headings found:')
for c in chunks_docaware:
    print(f'  [{c.heading_level}] {c.heading!r:50s}  {c.char_length} chars')
print()
print('--- First chunk (text_with_heading for embedding) ---')
print(chunks_docaware[0].text_with_heading[:500] if chunks_docaware else 'No chunks')

## Part 4 — LangChain Equivalents

The same three strategies in LangChain, all returning `list[Document]`:

| Strategy | From Scratch | LangChain |
|---------|-------------|----------|
| Fixed-size | `split_fixed_size()` | `RecursiveCharacterTextSplitter` |
| Semantic | `split_semantic()` | `SemanticChunker` (experimental) |
| Document-aware | `split_document_aware()` | `MarkdownHeaderTextSplitter` + char splitter |

In [ ]:
from src.langchain_impl.ingestion.loaders import load_pdf
from src.langchain_impl.ingestion.splitters import split_fixed_langchain

lc_docs = load_pdf(pdf_path, metadata={'client_id': 'acme_corp', 'doc_type': 'financial_report', 'year': 2023})
lc_chunks_fixed = split_fixed_langchain(lc_docs, chunk_size=512, chunk_overlap=64)

print(f'LangChain fixed-size: {len(lc_chunks_fixed)} chunks')
print(f'From scratch:         {len(chunks_fixed)} chunks')
print()
print('--- LangChain chunk 0 ---')
print(f'Content : {lc_chunks_fixed[0].page_content[:200]}')
print(f'Metadata: {lc_chunks_fixed[0].metadata}')

In [ ]:
from src.langchain_impl.ingestion.splitters import split_document_aware_langchain

lc_chunks_doc = split_document_aware_langchain(
    lc_docs,
    headers_to_split_on=[("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")],
    chunk_size=1500,
)

print(f'LangChain doc-aware: {len(lc_chunks_doc)} chunks')
print(f'From scratch:        {len(chunks_docaware)} chunks')
print()
for c in lc_chunks_doc[:4]:
    h1 = c.metadata.get('Header 1', '')
    h2 = c.metadata.get('Header 2', '')
    heading = h2 or h1 or '(no heading)'
    print(f'  {heading!r:40s}  {len(c.page_content)} chars')

## Part 5 — Comparison & Verdict

| | Fixed-size | Semantic | Document-aware |
|--|-----------|---------|---------------|
| **Speed** | Fast (no ML) | Slow (embeds sentences) | Fast (regex) |
| **Dependencies** | None | sentence-transformers | None |
| **Chunk size** | Predictable | Variable | Variable |
| **Heading context** | Lost | Lost | Preserved |
| **Table handling** | Mixes table text with prose | Mixes | Mixes |
| **Best for** | Homogeneous prose | Narrative with topic shifts | Structured docs |

### Production recommendation for this project

- **Financial reports (with tables):** `document_aware` for text sections + `table_parser` for tables (stored as separate chunks)
- **Contracts and NDAs:** `document_aware` — sections are critical ("3. Confidentiality" vs "5. Termination")
- **Policy documents:** `document_aware` — headings define scope
- **Unstructured narrative text:** `semantic` or `fixed_size`

**The heading matters more than you think.** If retrieval returns a chunk saying *"it must not exceed 30 days"* — is that a payment term or a termination notice period? Without the heading, nobody knows.

In [ ]:
print('=' * 60)
print('  CHUNKING STRATEGY COMPARISON')
print('=' * 60)
print(f'{"Strategy":<20} {"Chunks":>8} {"Avg chars":>10} {"Min":>6} {"Max":>6}')
print('-' * 60)

for name, chunks in [
    ('Fixed-size', chunks_fixed),
    ('Semantic', chunks_semantic),
    ('Document-aware', chunks_docaware),
]:
    if chunks:
        sizes = [c.char_length for c in chunks]
        print(f'{name:<20} {len(chunks):>8} {sum(sizes)/len(sizes):>10.0f} {min(sizes):>6} {max(sizes):>6}')

print('=' * 60)
print()
print('Next: Notebook 03 — Embeddings, Qdrant, and Naive RAG')

## What's Next

We now have three chunking strategies and a clear sense of when each one wins.

In **Notebook 03** we:
1. Embed all chunks with `text-embedding-3-small`
2. Store vectors + metadata in Qdrant
3. Build the first end-to-end RAG: embed query → retrieve chunks → generate answer
4. Test with 10-15 real questions and document where the system fails

That failure analysis is the setup for Notebook 04 (hybrid search + reranking).

---
*Article: «Как искать по этим кускам?»*